In [1]:
!pip install ultralytics insightface opencv-python-headless onnxruntime streamlit pyngrok -q
print("All packages installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 6.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 81.6 MB/s eta 0:00:00
All packages installed!


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/face_tracker/logs/entries', exist_ok=True)
os.makedirs('/content/face_tracker/logs/exits', exist_ok=True)
print("Folders ready!")

Mounted at /content/drive
Folders ready!


In [3]:
import json

config = {
    "detection_skip_frames": 5,
    "similarity_threshold": 0.55,
    "stability_frames": 1,
    "video_folder": "/content/drive/MyDrive/Video Datasets",
    "db_path": "/content/face_tracker/visitors.db",
    "log_dir": "/content/face_tracker/logs",
    "exit_timeout_frames": 30,
    "rtsp_url": ""
}

with open('/content/face_tracker/config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(" Config created!")
print(json.dumps(config, indent=2))

 Config created!
{
  "detection_skip_frames": 5,
  "similarity_threshold": 0.55,
  "stability_frames": 1,
  "video_folder": "/content/drive/MyDrive/Video Datasets",
  "db_path": "/content/face_tracker/visitors.db",
  "log_dir": "/content/face_tracker/logs",
  "exit_timeout_frames": 30,
  "rtsp_url": ""
}


In [4]:
import sqlite3
import json as jsonlib
import numpy as np
import shutil
import os

# Clean slate — remove old data so results are fresh
for path in [
    '/content/face_tracker/visitors.db',
    '/content/drive/MyDrive/visitors.db'
]:
    if os.path.exists(path):
        os.remove(path)
        print(f" Cleared: {path}")

if os.path.exists('/content/face_tracker/logs'):
    shutil.rmtree('/content/face_tracker/logs')

os.makedirs('/content/face_tracker/logs/entries', exist_ok=True)
os.makedirs('/content/face_tracker/logs/exits', exist_ok=True)
print(" Clean slate ready!")


def init_db(db_path):
    conn = sqlite3.connect(db_path)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS visitors (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            face_id     TEXT,
            event_type  TEXT,
            timestamp   TEXT,
            image_path  TEXT,
            video_file  TEXT
        )
    ''')
    conn.execute('''
        CREATE TABLE IF NOT EXISTS unique_visitors (
            face_id      TEXT PRIMARY KEY,
            first_seen   TEXT,
            last_seen    TEXT,
            total_visits INTEGER DEFAULT 1
        )
    ''')
    conn.execute('''
        CREATE TABLE IF NOT EXISTS embeddings (
            face_id   TEXT PRIMARY KEY,
            embedding TEXT
        )
    ''')
    conn.commit()
    return conn


def db_log(conn, face_id, event_type, timestamp, image_path, video_file):
    conn.execute(
        "INSERT INTO visitors (face_id, event_type, timestamp, image_path, video_file) "
        "VALUES (?,?,?,?,?)",
        (face_id, event_type, timestamp, image_path, video_file)
    )
    conn.commit()


def db_register_unique(conn, face_id, timestamp):
    conn.execute(
        "INSERT OR IGNORE INTO unique_visitors (face_id, first_seen, last_seen) "
        "VALUES (?,?,?)",
        (face_id, timestamp, timestamp)
    )
    conn.execute(
        "UPDATE unique_visitors SET last_seen=?, total_visits=total_visits+1 "
        "WHERE face_id=?",
        (timestamp, face_id)
    )
    conn.commit()


def save_embedding(conn, face_id, embedding):
    emb_str = jsonlib.dumps(embedding.tolist())
    conn.execute(
        "INSERT OR REPLACE INTO embeddings (face_id, embedding) VALUES (?,?)",
        (face_id, emb_str)
    )
    conn.commit()


def load_embeddings(conn):
    try:
        rows = conn.execute(
            "SELECT face_id, embedding FROM embeddings"
        ).fetchall()
        loaded = {}
        for face_id, emb_str in rows:
            loaded[face_id] = np.array(jsonlib.loads(emb_str))
        if loaded:
            print(f" Loaded {len(loaded)} existing embeddings from DB!")
        else:
            print(" No existing embeddings — starting fresh!")
        return loaded
    except Exception as e:
        print(f" Starting fresh! ({e})")
        return {}


def get_unique_count(conn):
    row = conn.execute("SELECT COUNT(*) FROM unique_visitors").fetchone()
    return row[0] if row else 0


conn = init_db('/content/face_tracker/visitors.db')
print(" Database ready!")

 Cleared: /content/drive/MyDrive/visitors.db
 Clean slate ready!
 Database ready!


In [5]:
from ultralytics import YOLO
import cv2
import urllib.request

class YOLOFaceDetector:
    """
    YOLOv8-based face detector.
    Downloads a dedicated face model from HuggingFace.
    Falls back to generic YOLOv8n (person + head crop) if download fails.
    """
    def __init__(self):
        model_path = '/content/face_tracker/yolov8n-face.pt'
        if not os.path.exists(model_path):
            print("Downloading YOLOv8 face model from HuggingFace...")
            try:
                urllib.request.urlretrieve(
                    'https://huggingface.co/arnabdhar/YOLOv8-Face-Detection'
                    '/resolve/main/model.pt',
                    model_path
                )
                size_mb = os.path.getsize(model_path) / (1024 * 1024)
                if size_mb < 1:
                    raise Exception("File too small — likely a redirect page")
                print(f"Downloaded: {size_mb:.1f} MB")
            except Exception as e:
                print(f"Download failed ({e}), using YOLOv8n fallback...")
                model_path = 'yolov8n.pt'

        self.model         = YOLO(model_path)
        self.is_face_model = 'yolov8n.pt' not in model_path
        print(f"YOLO loaded! Dedicated face model: {self.is_face_model}")

    def detect(self, frame):
        if self.is_face_model:
            results = self.model(frame, verbose=False, conf=0.5)
        else:
            # Generic model — detect persons, crop top 35% as face region
            results = self.model(frame, verbose=False, conf=0.5, classes=[0])

        boxes = []
        for r in results:
            for box in r.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                if not self.is_face_model:
                    h  = y2 - y1
                    y2 = y1 + int(h * 0.35)
                # Small padding around detected face
                x1 = max(0, x1 - 5)
                y1 = max(0, y1 - 5)
                x2 = min(frame.shape[1], x2 + 5)
                y2 = min(frame.shape[0], y2 + 5)
                if x2 > x1 and y2 > y1:
                    boxes.append((x1, y1, x2, y2))
        return boxes


yolo = YOLOFaceDetector()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Downloaded: 6.0 MB
YOLO loaded! Dedicated face model: True


In [6]:
import insightface
import numpy as np

class FaceRecognizer:
    """
    InsightFace-based face embedding extractor and matcher.

    Role in this system:
    - Primary role: generate embeddings for re-identification
      across track breaks (person left frame and re-entered).
    - Secondary role: store embeddings in DB for audit/analysis.

    Note: The tracker (Cell 7) handles re-ID within a continuous
    track. Embeddings handle re-ID across track breaks only.

    Embedding quality improvement:
    - Upscales small CCTV crops to 112x112 before embedding.
    - Applies light sharpening to improve edge definition.
    - Falls back to raw crop if upscaled version fails.
    """
    def __init__(self, conn):
        self.app = insightface.app.FaceAnalysis(
            providers=['CPUExecutionProvider']
        )
        self.app.prepare(ctx_id=0, det_size=(320, 320))
        self.known_faces = load_embeddings(conn)
        print(" InsightFace loaded!")

    def get_embedding(self, face_crop):
        """Extract a 512-dim face embedding from a crop."""
        if face_crop.shape[0] < 20 or face_crop.shape[1] < 20:
            return None
        try:
            # Upscale + sharpen for better embedding on small CCTV crops
            upscaled  = cv2.resize(face_crop, (112, 112),
                                   interpolation=cv2.INTER_CUBIC)
            kernel    = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
            sharpened = cv2.filter2D(upscaled, -1, kernel)

            faces = self.app.get(sharpened)
            if faces:
                return faces[0].embedding
            # Fallback: try original crop
            faces = self.app.get(face_crop)
            if faces:
                return faces[0].embedding
        except Exception:
            pass
        return None

    def find_match(self, embedding, threshold):
        """
        Cosine similarity search against all known face embeddings.
        Returns face_id if best match >= threshold, else None.
        Threshold 0.55 balances recall vs false positives for CCTV.
        """
        best_sim = -1
        best_id  = None
        for face_id, known_emb in self.known_faces.items():
            sim = float(
                np.dot(embedding, known_emb) /
                (np.linalg.norm(embedding) * np.linalg.norm(known_emb))
            )
            if sim > best_sim:
                best_sim = sim
                best_id  = face_id
        if best_sim >= threshold:
            return best_id
        return None

    def register(self, face_id, embedding):
        """Add a new face embedding to the in-memory store."""
        self.known_faces[face_id] = embedding

    def update_embedding(self, face_id, new_embedding):
        """
        Update stored embedding using exponential moving average.
        Keeps the embedding fresh as face angle/lighting changes.
        alpha=0.3 means 30% new, 70% historical — stable but adaptive.
        """
        if face_id in self.known_faces:
            alpha = 0.3
            updated = (
                alpha * new_embedding +
                (1 - alpha) * self.known_faces[face_id]
            )
            # Re-normalise to unit vector for cosine similarity
            self.known_faces[face_id] = (
                updated / np.linalg.norm(updated)
            )
        else:
            self.register(face_id, new_embedding)


recognizer = FaceRecognizer(conn)



download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:07<00:00, 37124.91KB/s]


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (320, 320)
 No existing

In [7]:
class CentroidTracker:
    """
    Tracks faces across frames within a video using centroid
    distance combined with IoU overlap scoring.

    Why not DeepSort/ByteTrack:
    - This is a single fixed camera with sparse foot traffic.
    - Centroid + IoU is lightweight, CPU-friendly, and accurate
      enough for non-crowded single-camera scenes.
    - DeepSort adds appearance model overhead not needed here.

    Key design decisions:
    - stability_frames gate: a track must be alive for N frames
      before being registered as a visitor. Filters out brief
      blurry detections that would inflate the count.
    - missing counter only increments when detection actually ran
      (not on skipped frames), preventing premature exits.
    - last_crop saved every matched frame for reliable exit images.
    """
    def __init__(self, exit_timeout=30, stability_frames=8):
        self.tracks           = {}
        self.next_id          = 0
        self.exit_timeout     = exit_timeout
        self.stability_frames = stability_frames

    def _centroid(self, box):
        x1, y1, x2, y2 = box
        return ((x1 + x2) // 2, (y1 + y2) // 2)

    def _dist(self, c1, c2):
        return ((c1[0] - c2[0]) ** 2 + (c1[1] - c2[1]) ** 2) ** 0.5

    def _iou(self, a, b):
        xA = max(a[0], b[0]); yA = max(a[1], b[1])
        xB = min(a[2], b[2]); yB = min(a[3], b[3])
        inter = max(0, xB - xA) * max(0, yB - yA)
        if inter == 0:
            return 0.0
        area_a = (a[2] - a[0]) * (a[3] - a[1])
        area_b = (b[2] - b[0]) * (b[3] - b[1])
        return inter / float(area_a + area_b - inter)

    def _score(self, track, cent, box):
        """
        Combined matching score.
        Low centroid distance + high IoU = good match (low score).
        """
        dist = self._dist(track['centroid'], cent)
        iou  = self._iou(track['box'], box)
        return dist * (1 - iou * 0.5)

    def update(self, boxes, frame, detection_ran):
        """
        Args:
            boxes         : list of (x1,y1,x2,y2) from YOLO
            frame         : current video frame (for crop saving)
            detection_ran : True if YOLO ran this frame

        Returns:
            active_tracks : dict of tid -> track info
            new_track_ids : list of tids that just appeared
            exited        : list of (tid, face_id, exit_crop)
        """
        exited      = []
        current     = [(self._centroid(b), b) for b in boxes]
        matched_det = set()

        # ── Match detections to existing tracks ──────────────
        for tid, track in self.tracks.items():
            best_score = 80   # distance threshold — tune if needed
            best_i     = None
            for i, (cent, box) in enumerate(current):
                if i in matched_det:
                    continue
                score = self._score(track, cent, box)
                if score < best_score:
                    best_score = score
                    best_i     = i

            if best_i is not None:
                cent, box         = current[best_i]
                track['centroid'] = cent
                track['box']      = box
                track['missing']  = 0
                track['age']      = track.get('age', 0) + 1
                x1, y1, x2, y2   = box
                crop              = frame[y1:y2, x1:x2]
                if crop.size > 0:
                    track['last_crop'] = crop.copy()
                matched_det.add(best_i)
            else:
                # Only penalise when detection actually ran
                if detection_ran:
                    track['missing'] += 1

        # ── Register unmatched detections as new tracks ───────
        new_track_ids = []
        for i, (cent, box) in enumerate(current):
            if i not in matched_det:
                x1, y1, x2, y2 = box
                crop = frame[y1:y2, x1:x2]
                self.tracks[self.next_id] = {
                    'centroid':  cent,
                    'box':       box,
                    'face_id':   None,
                    'missing':   0,
                    'age':       1,
                    'last_crop': crop.copy() if crop.size > 0 else None
                }
                new_track_ids.append(self.next_id)
                self.next_id += 1

        # ── Declare exits for long-missing tracks ─────────────
        to_remove = []
        for tid, track in self.tracks.items():
            if track['missing'] >= self.exit_timeout:
                exited.append((tid, track['face_id'], track.get('last_crop')))
                to_remove.append(tid)
        for tid in to_remove:
            del self.tracks[tid]

        return self.tracks, new_track_ids, exited


print("Centroid + IoU Tracker ready!")




Centroid + IoU Tracker ready!


In [8]:
os.makedirs('/content/face_tracker/logs', exist_ok=True)
import logging
from datetime import datetime

logging.basicConfig(
    filename='/content/face_tracker/logs/events.log',
    level=logging.INFO,
    format='%(asctime)s - %(message)s'
)


def save_face_image(face_crop, face_id, event_type, log_dir):
    date_str = datetime.now().strftime("%Y-%m-%d")
    folder   = f'{log_dir}/{event_type}s/{date_str}'
    os.makedirs(folder, exist_ok=True)
    ts   = datetime.now().strftime("%H-%M-%S-%f")
    path = f'{folder}/{face_id}_{ts}.jpg'
    cv2.imwrite(path, face_crop)
    return path


def log_entry(face_id, img_path, video_file):
    logging.info(
        f"ENTRY               | face_id={face_id} | "
        f"video={video_file} | img={img_path}"
    )

def log_exit(face_id, img_path, video_file):
    logging.info(
        f"EXIT                | face_id={face_id} | "
        f"video={video_file} | img={img_path}"
    )

def log_embedding(face_id, video_file):
    logging.info(
        f"EMBEDDING_GENERATED | face_id={face_id} | video={video_file}"
    )

def log_tracking(face_id, track_id, video_file):
    logging.info(
        f"TRACKING            | face_id={face_id} | "
        f"track_id={track_id} | video={video_file}"
    )

def log_reidentified(face_id, video_file):
    logging.info(
        f"RE_IDENTIFIED       | face_id={face_id} | video={video_file}"
    )


print(" Logger ready!")



 Logger ready!


In [9]:
import uuid
from datetime import datetime

with open('/content/face_tracker/config.json') as f:
    config = json.load(f)

rtsp_url         = config.get('rtsp_url', '').strip()
video_folder     = config['video_folder']
skip             = config['detection_skip_frames']
similarity_thr   = config['similarity_threshold']
stability_frames = config['stability_frames']

if rtsp_url:
    all_videos = ['LIVE_RTSP']
    print(f" RTSP mode: {rtsp_url}")
else:
    all_videos = sorted([
        f for f in os.listdir(video_folder) if f.endswith('.mp4')
    ])
    print(f" Found {len(all_videos)} video(s)\n")

frame_count = 0

for video_name in all_videos:

    if rtsp_url:
        cap = cv2.VideoCapture(rtsp_url)
    else:
        cap = cv2.VideoCapture(os.path.join(video_folder, video_name))

    if not cap.isOpened():
        print(f" Could not open: {video_name}")
        continue

    print(f" Processing: {video_name}")
    video_frame   = 0
    last_boxes    = []
    detection_ran = False

    tracker = CentroidTracker(
        exit_timeout=config['exit_timeout_frames'],
        stability_frames=stability_frames
    )

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        video_frame += 1

        # ── Step 1: YOLO Detection every N frames ────────────
        detection_ran = (frame_count % skip == 0)
        if detection_ran:
            last_boxes = yolo.detect(frame)
        boxes = last_boxes  # reuse last known boxes on skipped frames

        # ── Step 2: Update Tracker ────────────────────────────
        active_tracks, new_track_ids, exited = tracker.update(
            boxes, frame, detection_ran
        )

        # ── Step 3: Handle New Tracks (with stability gate) ───
        for tid in new_track_ids:
            track = active_tracks.get(tid)
            if track is None:
                continue

            # Stability gate: skip tracks that are too new/brief.
            # They haven't been alive long enough to have a quality
            # crop — likely a blurry or partial detection.
            # The track stays alive in the tracker and will be
            # registered on the next detection cycle if it persists.
            if track.get('age', 1) < stability_frames:
                continue

            crop = track.get('last_crop')
            if crop is None or crop.size == 0:
                continue

            # Get embedding for re-ID check across track breaks
            embedding = recognizer.get_embedding(crop)
            log_embedding(f"track_{tid}", video_name)

            # Check if this is a re-entering known person
            matched_face_id = None
            if embedding is not None:
                matched_face_id = recognizer.find_match(
                    embedding, similarity_thr
                )

            if matched_face_id is not None:
                # ── Known person re-entered ───────────────────
                # Assign same face_id, log entry, but DO NOT
                # increment unique visitor count.
                face_id          = matched_face_id
                track['face_id'] = face_id

                # Update embedding with new angle/lighting data
                if embedding is not None:
                    recognizer.update_embedding(face_id, embedding)
                    save_embedding(conn, face_id,
                                   recognizer.known_faces[face_id])

                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                img_path  = save_face_image(
                    crop, face_id, 'entry', config['log_dir']
                )
                db_log(conn, face_id, 'entry', timestamp,
                       img_path, video_name)
                log_reidentified(face_id, video_name)
                print(f"   RE-ENTRY: {face_id} (not counted again)")

            else:
                # ── Genuinely new person ──────────────────────
                face_id          = str(uuid.uuid4())[:8]
                track['face_id'] = face_id

                if embedding is not None:
                    recognizer.register(face_id, embedding)
                    save_embedding(conn, face_id, embedding)

                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                img_path  = save_face_image(
                    crop, face_id, 'entry', config['log_dir']
                )
                db_log(conn, face_id, 'entry', timestamp,
                       img_path, video_name)
                db_register_unique(conn, face_id, timestamp)
                log_entry(face_id, img_path, video_name)

                unique_count = get_unique_count(conn)
                print(
                    f"   NEW: {face_id} | "
                    f"Total unique: {unique_count}"
                )

        # ── Step 4: Log Active Tracking ───────────────────────
        for tid, track in active_tracks.items():
            if track['face_id']:
                log_tracking(track['face_id'], tid, video_name)

        # ── Step 5: Log Exits ─────────────────────────────────
        for tid, face_id, exit_crop in exited:
            if face_id is None:
                continue
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            exit_img  = ''
            if exit_crop is not None and exit_crop.size > 0:
                exit_img = save_face_image(
                    exit_crop, face_id, 'exit', config['log_dir']
                )
            db_log(conn, face_id, 'exit', timestamp,
                   exit_img, video_name)
            log_exit(face_id, exit_img, video_name)
            print(f"   EXIT: {face_id}")

    # ── End of video: log remaining tracks as exits ───────────
    for tid, track in tracker.tracks.items():
        if track['face_id']:
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            exit_img  = ''
            crop      = track.get('last_crop')
            if crop is not None and crop.size > 0:
                exit_img = save_face_image(
                    crop, track['face_id'], 'exit', config['log_dir']
                )
            db_log(conn, track['face_id'], 'exit',
                   timestamp, exit_img, video_name)
            log_exit(track['face_id'], exit_img, video_name)

    cap.release()
    print(f"   Done: {video_name} | Frames processed: {video_frame}\n")

print(f"\n ALL DONE!")
print(f" Total Unique Visitors: {get_unique_count(conn)}")

print("\n PER VIDEO SUMMARY:")
print("-" * 50)
rows = conn.execute("""
    SELECT video_file, COUNT(*) as entries
    FROM visitors
    WHERE event_type = 'entry'
    GROUP BY video_file
    ORDER BY video_file
""").fetchall()
for video, count in rows:
    print(f"  {video}: {count} entries")
print("-" * 50)


 Found 23 video(s)

 Processing: record_20250620_183903.mp4
   NEW: 8dbaf736 | Total unique: 1
   EXIT: 8dbaf736
   NEW: 7ab23b3d | Total unique: 2
   NEW: 3971c77f | Total unique: 3
   NEW: b22515a3 | Total unique: 4
   NEW: 77e0e312 | Total unique: 5
   NEW: 24987ecb | Total unique: 6
   EXIT: 7ab23b3d
   EXIT: b22515a3
   EXIT: 77e0e312
   NEW: 94bb8cef | Total unique: 7
   NEW: d7dd0e26 | Total unique: 8
   RE-ENTRY: 94bb8cef (not counted again)
   RE-ENTRY: d7dd0e26 (not counted again)
   RE-ENTRY: d7dd0e26 (not counted again)
   RE-ENTRY: 94bb8cef (not counted again)
   RE-ENTRY: 94bb8cef (not counted again)
   RE-ENTRY: 94bb8cef (not counted again)
   NEW: 92f16944 | Total unique: 9
   NEW: ce0c9b40 | Total unique: 10
   RE-ENTRY: ce0c9b40 (not counted again)
   RE-ENTRY: ce0c9b40 (not counted again)
   RE-ENTRY: ce0c9b40 (not counted again)
   EXIT: 24987ecb
   EXIT: 3971c77f
   EXIT: 94bb8cef
   EXIT: d7dd0e26
   EXIT: 94bb8cef
   EXIT: d7dd0e26
   EXIT: d7dd0e26
   EXIT: 94bb

In [19]:
import shutil

shutil.copy(
    '/content/face_tracker/visitors.db',
    '/content/drive/MyDrive/visitors.db'
)
shutil.copytree(
    '/content/face_tracker/logs',
    '/content/drive/MyDrive/face_tracker_logs',
    dirs_exist_ok=True
)
shutil.copy(
    '/content/face_tracker/config.json',
    '/content/drive/MyDrive/config.json'
)
print(" DB, logs, and config saved to Drive!")



 DB, logs, and config saved to Drive!


In [20]:
import json as jsonlib

embeddings_backup = {}
for face_id, emb in recognizer.known_faces.items():
    embeddings_backup[face_id] = emb.tolist()

with open('/content/drive/MyDrive/embeddings_backup.json', 'w') as f:
    jsonlib.dump(embeddings_backup, f)

print(f" Saved {len(embeddings_backup)} embeddings to Drive!")

 Saved 138 embeddings to Drive!


In [21]:
import pandas as pd

print("=" * 60)
print("              VERIFICATION REPORT")
print("=" * 60)

df  = pd.read_sql_query("SELECT * FROM visitors", conn)
uv  = pd.read_sql_query("SELECT * FROM unique_visitors", conn)
emb = pd.read_sql_query("SELECT COUNT(*) as count FROM embeddings", conn)

print(f"\n DATABASE CHECK")
print(f"  Total events logged  : {len(df)}")
print(f"  Unique visitors      : {get_unique_count(conn)}")
print(f"  Entry events         : {len(df[df['event_type']=='entry'])}")
print(f"  Exit events          : {len(df[df['event_type']=='exit'])}")
print(f"  Saved embeddings     : {emb['count'][0]}")
print(f"  Videos covered       : {df['video_file'].nunique()}")

missing = sum(
    1 for _, row in df[df['event_type'] == 'entry'].iterrows()
    if not row['image_path'] or not os.path.exists(str(row['image_path']))
)
exit_imgs = sum(
    1 for _, row in df[df['event_type'] == 'exit'].iterrows()
    if row['image_path'] and os.path.exists(str(row['image_path']))
)
print(f"\n  IMAGE CHECK")
print(f"  Entry images saved   : {len(df[df['event_type']=='entry']) - missing}")
print(f"  Missing entry images : {missing}")
print(f"  Exit images saved    : {exit_imgs}")

log_path = '/content/face_tracker/logs/events.log'
if os.path.exists(log_path):
    with open(log_path) as f:
        lines = f.readlines()
    print(f"\n LOG FILE CHECK")
    print(f"  Total log lines          : {len(lines)}")
    print(f"  ENTRY                    : {sum(1 for l in lines if 'ENTRY' in l)}")
    print(f"  EXIT                     : {sum(1 for l in lines if 'EXIT' in l)}")
    print(f"  TRACKING                 : {sum(1 for l in lines if 'TRACKING' in l)}")
    print(f"  EMBEDDING_GENERATED      : {sum(1 for l in lines if 'EMBEDDING' in l)}")
    print(f"  RE_IDENTIFIED            : {sum(1 for l in lines if 'RE_IDENTIFIED' in l)}")
else:
    print("\n Log file not found!")

entry_counts = df[df['event_type'] == 'entry']['face_id'].value_counts()
duplicates   = entry_counts[entry_counts > 1]
print(f"\n DUPLICATE CHECK")
if len(duplicates) == 0:
    print(f"   No face registered twice as unique!")
else:
    print(f"    {len(duplicates)} face IDs appear in more than one entry")
    print(f"     (These are re-entries — expected behaviour)")

print(f"\n SAMPLE ENTRIES (first 5)")
print(df[df['event_type'] == 'entry'][
    ['face_id', 'timestamp', 'video_file']
].head().to_string())

print(f"\n{'=' * 60}")
print(" VERIFICATION COMPLETE")
print("=" * 60)



              VERIFICATION REPORT

 DATABASE CHECK
  Total events logged  : 538
  Unique visitors      : 172
  Entry events         : 269
  Exit events          : 269
  Saved embeddings     : 138
  Videos covered       : 22

  IMAGE CHECK
  Entry images saved   : 269
  Missing entry images : 0
  Exit images saved    : 269

 LOG FILE CHECK
  Total log lines          : 677
  ENTRY                    : 269
  EXIT                     : 269
  TRACKING                 : 0
  EMBEDDING_GENERATED      : 138
  RE_IDENTIFIED            : 0

 DUPLICATE CHECK
    34 face IDs appear in more than one entry
     (These are re-entries — expected behaviour)

 SAMPLE ENTRIES (first 5)
    face_id            timestamp                  video_file
0  8dbaf736  2026-03-22 10:02:45  record_20250620_183903.mp4
2  7ab23b3d  2026-03-22 10:03:05  record_20250620_183903.mp4
3  3971c77f  2026-03-22 10:03:07  record_20250620_183903.mp4
4  b22515a3  2026-03-22 10:03:10  record_20250620_183903.mp4
5  77e0e312  2026-03

In [31]:
lines = [
    "import streamlit as st\n",
    "import sqlite3\n",
    "import pandas as pd\n",
    "import os\n",
    "from PIL import Image\n",
    "\n",
    "st.set_page_config(page_title='Face Tracker Dashboard', layout='wide')\n",
    "st.title('Intelligent Face Tracker Dashboard')\n",
    "st.markdown('YOLO + InsightFace + Centroid Tracker | Unique Visitor Counter')\n",
    "st.divider()\n",
    "\n",
    "DB_PATH = '/content/face_tracker/visitors.db'\n",
    "\n",
    "def load_data():\n",
    "    conn = sqlite3.connect(DB_PATH)\n",
    "    df = pd.read_sql_query('SELECT * FROM visitors ORDER BY timestamp ASC', conn)\n",
    "    uv = pd.read_sql_query('SELECT * FROM unique_visitors ORDER BY first_seen', conn)\n",
    "    conn.close()\n",
    "    return df, uv\n",
    "\n",
    "df, uv = load_data()\n",
    "\n",
    "c1, c2, c3, c4 = st.columns(4)\n",
    "c1.metric('Unique Visitors', len(uv))\n",
    "c2.metric('Videos Processed', df['video_file'].nunique())\n",
    "c3.metric('Entry Events', len(df[df['event_type'] == 'entry']))\n",
    "c4.metric('Exit Events', len(df[df['event_type'] == 'exit']))\n",
    "st.divider()\n",
    "\n",
    "st.subheader('Filter by Video')\n",
    "vids = ['All Videos'] + sorted(df['video_file'].unique().tolist())\n",
    "sel_vid = st.selectbox('Select Video', vids)\n",
    "fdf = df.copy()\n",
    "if sel_vid != 'All Videos':\n",
    "    fdf = fdf[fdf['video_file'] == sel_vid]\n",
    "st.divider()\n",
    "\n",
    "st.subheader('Detected Faces Gallery')\n",
    "visible_faces = (\n",
    "    fdf[fdf['event_type'] == 'entry']\n",
    "    .sort_values('timestamp')\n",
    "    .drop_duplicates(subset='face_id', keep='first')\n",
    "    .head(12)\n",
    ")\n",
    "cols = st.columns(6)\n",
    "shown = 0\n",
    "for _, row in visible_faces.iterrows():\n",
    "    path = str(row['image_path']) if row['image_path'] else ''\n",
    "    if path and os.path.exists(path):\n",
    "        with cols[shown % 6]:\n",
    "            img = Image.open(path)\n",
    "            fid = row['face_id'][:6]\n",
    "            st.image(img, caption='ID: ' + fid, use_container_width=True)\n",
    "        shown += 1\n",
    "    if shown >= 12:\n",
    "        break\n",
    "if shown == 0:\n",
    "    st.info('No face images found.')\n",
    "st.divider()\n",
    "\n",
    "st.subheader('Visitor Timeline by Hour')\n",
    "if not fdf.empty:\n",
    "    plot_df = fdf.copy()\n",
    "    plot_df['hour'] = pd.to_datetime(plot_df['timestamp'], errors='coerce').dt.hour\n",
    "    tl = (\n",
    "        plot_df[plot_df['event_type'] == 'entry']\n",
    "        .drop_duplicates(subset='face_id')\n",
    "        .groupby('hour')\n",
    "        .size()\n",
    "        .reset_index(name='visitors')\n",
    "    )\n",
    "    if not tl.empty:\n",
    "        st.bar_chart(tl.set_index('hour'))\n",
    "    else:\n",
    "        st.info('No data to plot.')\n",
    "st.divider()\n",
    "\n",
    "st.subheader('Event Log')\n",
    "st.dataframe(\n",
    "    fdf[fdf['event_type'] == 'entry'][['face_id', 'timestamp', 'video_file']],\n",
    "    use_container_width=True,\n",
    "    height=350\n",
    ")\n",
    "\n",
    "st.subheader('Unique Visitors Registry')\n",
    "st.dataframe(uv, use_container_width=True, height=300)\n",
    "st.divider()\n",
    "\n",
    "csv = fdf.to_csv(index=False)\n",
    "st.download_button('Download Event Log as CSV', csv, 'visitor_log.csv', 'text/csv')\n",
    "st.markdown('---')\n",
    "st.caption('This project is a part of a hackathon run by https://katomaran.com')\n",
]

with open('/content/face_tracker/app.py', 'w') as f:
    f.writelines(lines)
print("app.py updated!")

app.py updated!


In [32]:
ngrok_token = input("Paste your ngrok authtoken: ")
!ngrok authtoken {ngrok_token}

from pyngrok import ngrok
import subprocess, time

subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)

proc = subprocess.Popen([
    'streamlit', 'run', '/content/face_tracker/app.py',
    '--server.port', '8501',
    '--server.headless', 'true'
])
time.sleep(5)

url = ngrok.connect(8501)
print(f"\n Dashboard is LIVE at: {url}")
print(" Click the link above to open your dashboard!")

Paste your ngrok authtoken: 3BFseF5xhKgIjnaK2t6sdZV2fdQ_4LoYTm5xqPXssiZJhhUh4
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml

 Dashboard is LIVE at: NgrokTunnel: "https://oda-extinct-allegra.ngrok-free.dev" -> "http://localhost:8501"
 Click the link above to open your dashboard!


In [ ]:
# Run this instead of Cell 9 on future restarts
import shutil, json as jsonlib, numpy as np

shutil.copy('/content/drive/MyDrive/visitors.db',
            '/content/face_tracker/visitors.db')
shutil.copytree('/content/drive/MyDrive/face_tracker_logs',
                '/content/face_tracker/logs',
                dirs_exist_ok=True)

with open('/content/drive/MyDrive/embeddings_backup.json') as f:
    backup = jsonlib.load(f)
recognizer.known_faces = {k: np.array(v) for k, v in backup.items()}
conn = init_db('/content/face_tracker/visitors.db')

print(f"Restored {len(recognizer.known_faces)} embeddings!")
print(f" Unique visitors: {get_unique_count(conn)}")
print(" Ready!")